# Do WER: sua dau vao ASR (`docs/asr-feature-fix-design.md`)

**Cau hoi:** engine da sua (mel tinh chong audio, giu pre-encode cache o buoc 2) co WER khong te hon engine cu khong?

**Truoc khi chay** (giong `kaggle_end_silence.ipynb`)

1. Session Settings: Accelerator **GPU T4 x2**, **Internet ON**.
2. Add Data hai Kaggle Dataset:
   - `models`: noi dung `models/` (xem `scripts/copy-models.sh`), cung file `.nemo` voi server.
   - `source`: zip cua repo khong co `models/`, `wheels_gpu/`, `.git`. **Zip tu working tree hien tai va THEM thu muc
     `tests`** (lenh zip trong `docs/scripts-guide.md` muc 8.3): o kiem mel that dung `tests/test_runtime_features.py`.
3. Sua `MODELS_DATASET`, `SRC_DATASET` o cell cau hinh roi Run All. `LIMIT = 20` de chay thu nhanh.

**Chay gi**

- **Kiem tra mel that** (`scripts/kaggle/check_mel.py`, truoc khi chay engine): dung preprocessor that tu file `.nemo` va NeMo cua venv (cung ban
  `2.8.0rc0` voi server). In cau hinh preprocessor va `get_seq_len` (muc 5 cua tai lieu), kiem cau hinh nhu engine
  luc nap model (Q2), roi so chunk dua vao model voi cach chia chunk cua NeMo tham chieu tren mel ca utterance.
  Sai thi dung o day.
- **Luot A** = engine truoc khi sua: buoc `B` cua `run_tests_kaggle.sh` (`language_source=vi`, 200 clip) voi
  `ASR_FEATURE_LEGACY=1`. Ket qua trong `out_legacy/`.
- **Luot F** = engine da sua: cung buoc `B`, khong co bien do. Ket qua trong `out_fixed/`.
- Moi luot mot lan khoi dong engine (nap lai model vai phut).

**Doc ket qua**

- Bang WER: base `A_legacy`, so voi `F_fixed`. `delta_pp` am = ban sua tot hon; chenh duoi ~1 diem % la nhieu, nhin
  `ci95_pp` (khoang tin cay 95%).
- Tieu chi (muc 6 cua tai lieu): can tren `ci95_pp` cua F - A <= 0,5 diem, va xem tay cac clip xau di (`diff`).
- `logstats`: so utterance moi clip va so cau ASR rong cua tung luot.

**Gioi han**

- T4 khong phai L4; gui nhanh hon thoi gian thuc nen **khong do duoc do tre** (ban sua khong lam STEP cham hon:
  da kiem bang test). Dataset la cau doc, khong phai hoi thoai dien thoai.


In [ ]:
import os, subprocess

MODELS_DATASET = "models"       # ten dataset trong /kaggle/input
SRC_DATASET = "source"          # ten dataset chua source (zip hoac thu muc da giai nen)

LIMIT = 0                       # 0 = du 200 clip; dat 20 de chay thu nhanh
CONCURRENCY = 4

ENGINE_DIR = "/kaggle/working/engine"
VENV = "/kaggle/working/venv"
PY = f"{VENV}/bin/python"
OUT_LEGACY = "/kaggle/working/out_legacy"   # luot A: ASR_FEATURE_LEGACY=1 (engine truoc khi sua)
OUT_FIXED = "/kaggle/working/out_fixed"     # luot F: engine da sua
OUT_AB = "/kaggle/working/out_ab"           # bang so sanh A vs F

def sh(cmd, check=True):
    print("$", cmd)
    proc = subprocess.run(["bash", "-lc", cmd], text=True)
    if check and proc.returncode != 0:
        raise RuntimeError(f"exit {proc.returncode}: {cmd}")

sh("ls /kaggle/input; nvidia-smi -L")

In [ ]:
# Tim thu muc dataset (Kaggle co the dat dataset duoi /kaggle/input/<ten> hoac /kaggle/input/datasets/<owner>/<ten>)
def find_dataset(name):
    hits = []
    for root, dirs, _ in os.walk("/kaggle/input"):
        if os.path.basename(root) == name:
            hits.append(root)
        if root.count(os.sep) >= 5:
            dirs[:] = []
    if not hits:
        raise FileNotFoundError(f"khong thay dataset '{name}' trong /kaggle/input")
    return sorted(hits, key=len)[0]

models_dir = find_dataset(MODELS_DATASET)
src_dir = find_dataset(SRC_DATASET)
print("models:", models_dir)
print("source:", src_dir)

In [ ]:
# Chep source sang noi ghi duoc (giai nen neu la zip)
sh(f"rm -rf {ENGINE_DIR} && mkdir -p {ENGINE_DIR}")
zips = [f for f in os.listdir(src_dir) if f.endswith(".zip")]
if zips:
    sh(f"cd {ENGINE_DIR} && unzip -q '{src_dir}/{zips[0]}'")
else:
    sh(f"cp -a '{src_dir}/.' {ENGINE_DIR}/")

# neu zip co thu muc goc duy nhat thi nhay vao trong
entries = os.listdir(ENGINE_DIR)
if "scripts" not in entries and len(entries) == 1:
    ENGINE_DIR = os.path.join(ENGINE_DIR, entries[0])
assert os.path.exists(f"{ENGINE_DIR}/scripts/kaggle/setup_kaggle.sh"), "khong thay scripts/kaggle trong source"
# source phai co ban sua va cong tac ASR_FEATURE_LEGACY; zip cu thi dung lai som
assert "ASR_FEATURE_LEGACY" in open(f"{ENGINE_DIR}/src/nemotron_testbed/runtime.py", encoding="utf-8").read(), \
    "runtime.py trong source chua co ban sua: zip lai tu working tree hien tai"
assert os.path.exists(f"{ENGINE_DIR}/tests/test_runtime_features.py"), \
    "source thieu tests/test_runtime_features.py: zip them thu muc tests (docs/scripts-guide.md muc 8.3)"
assert os.path.exists(f"{ENGINE_DIR}/scripts/kaggle/check_mel.py"), \
    "source chua co scripts/kaggle/check_mel.py: zip lai tu working tree hien tai"
NEMO_FILE = f"{ENGINE_DIR}/models/nemotron-3.5-asr-streaming-0.6b/nemotron-3.5-asr-streaming-0.6b.nemo"
sh(f"ls {ENGINE_DIR}")

In [ ]:
# Cai dat (vai chuc phut lan dau): venv Python 3.12, torch cu124, lock, piper-tts, patch NeMo
sh(f"cd {ENGINE_DIR} && MODELS_DIR='{models_dir}' bash scripts/kaggle/setup_kaggle.sh")

In [ ]:
# Kiem tra mel that (vai giay, scripts/kaggle/check_mel.py): cau hinh preprocessor cua .nemo, get_seq_len cua NeMo
# trong venv, kiem cau hinh nhu engine (Q2), chunk dua vao model == NeMo tham chieu. Loi o day thi dung, chua chay engine.
sh(f"cd {ENGINE_DIR} && source {VENV}/engine-env.sh && {PY} scripts/kaggle/check_mel.py '{NEMO_FILE}'")

In [ ]:
# Luot A: engine TRUOC khi sua (ASR_FEATURE_LEGACY=1), buoc B, mot lan khoi dong engine (vai phut)
common = f"STEPS=B LIMIT={LIMIT} CONCURRENCY={CONCURRENCY}"
sh(f"cd {ENGINE_DIR} && ASR_FEATURE_LEGACY=1 OUT_DIR={OUT_LEGACY} {common} bash scripts/kaggle/run_tests_kaggle.sh")

In [ ]:
# Luot F: engine DA sua (khong co ASR_FEATURE_LEGACY), cung buoc B va cung du lieu
sh(f"cd {ENGINE_DIR} && env -u ASR_FEATURE_LEGACY OUT_DIR={OUT_FIXED} {common} bash scripts/kaggle/run_tests_kaggle.sh")

In [ ]:
# Kiem tra moi luot chay dung duong mel: log luot A phai co dong ASR_FEATURE_LEGACY (moi ASR worker mot dong), luot F khong co
def legacy_lines(out_dir):
    path = f"{out_dir}/logs/main/nemotron-engine.log"
    return sum("ASR_FEATURE_LEGACY" in line for line in open(path, encoding="utf-8", errors="replace"))

print("A (legacy):", legacy_lines(OUT_LEGACY), "dong; F (fixed):", legacy_lines(OUT_FIXED), "dong")
assert legacy_lines(OUT_LEGACY) > 0, "luot A khong chay duong cu: kiem tra ASR_FEATURE_LEGACY"
assert legacy_lines(OUT_FIXED) == 0, "luot F van chay duong cu"

In [ ]:
# Ket qua: WER A (truoc khi sua) vs F (da sua), clip doi nhieu nhat, utterances theo luot
sh(f"mkdir -p {OUT_AB} && cp {OUT_LEGACY}/B_source_vi.jsonl {OUT_AB}/A_legacy.jsonl "
   f"&& cp {OUT_FIXED}/B_source_vi.jsonl {OUT_AB}/F_fixed.jsonl")
EVAL = f"{PY} {ENGINE_DIR}/scripts/kaggle/eval_wer.py"
sh(f"{EVAL} score --base {OUT_AB}/A_legacy.jsonl {OUT_AB}/F_fixed.jsonl --csv {OUT_AB}/score_feature_fix.csv")
sh(f"{EVAL} diff --base {OUT_AB}/A_legacy.jsonl {OUT_AB}/F_fixed.jsonl -n 8", check=False)
for name, out_dir in (("A legacy", OUT_LEGACY), ("F fixed", OUT_FIXED)):
    print("=" * 30, name)
    sh(f"{EVAL} logstats --tags B {out_dir}/logs/main/nemotron-engine.log", check=False)